In [1]:
import re
import numpy as np

def parse_manual_castep(filename):
    with open(filename, 'r') as f:
        lines = f.readlines()

    lattice = []
    species = []
    coords_frac = []
    forces = []
    energy = None
    
    # State flags
    reading_lattice = False
    reading_positions = False
    reading_forces = False

    for line in lines:
        # 0. Clean the line of source tags and table borders 
        clean = re.sub(r"\\", "", line).strip()
        
        # 1. Extract Energy 
        if "Final energy, E" in clean:
            energy_match = re.search(r"=\s+(-?\d+\.\d+)", clean)
            if energy_match:
                energy = float(energy_match.group(1))

        # 2. Extract Lattice 
        if "Real Lattice(A)" in clean:
            reading_lattice = True
            continue
        if reading_lattice:
            parts = clean.split()
            if len(parts) >= 3 and len(lattice) < 3:
                lattice.append([float(x) for x in parts[:3]])
            elif len(lattice) == 3:
                reading_lattice = False

        # 3. Extract Fractional Positions 
        if "Fractional coordinates of atoms" in clean:
            reading_positions = True
            continue
        if reading_positions:
            # Skip the table header lines
            if "x---" in clean or "Element" in clean:
                continue
            # End of table marker
            if "xxxxxxxxxx" in clean or "No user defined" in clean:
                reading_positions = False
                continue
            
            parts = clean.replace('x', '').split()
            if len(parts) >= 5:
                species.append(parts[0]) # Element 
                coords_frac.append([float(parts[2]), float(parts[3]), float(parts[4])]) # u, v, w 

        # 4. Extract Forces 
        if "Cartesian components (eV/A)" in clean:
            reading_forces = True
            continue
        if reading_forces:
            if "---" in clean or "x  y  z" in clean:
                continue
            if "*" not in clean: # End of force block
                reading_forces = False
                continue
            
            parts = clean.replace('*', '').split()
            if len(parts) >= 4:
                # The last 3 parts are always the x, y, z components 
                forces.append([float(parts[-3]), float(parts[-2]), float(parts[-1])])

    # Convert to numpy and perform coordinate transformation 
    lattice_np = np.array(lattice)
    coords_np = np.array(coords_frac)
    
    # This matmul will now succeed because lattice_np is (3,3) 
    positions_abs = coords_np @ lattice_np 
    forces_np = np.array(forces)

    return {
        "energy": energy, 
        "lattice": lattice_np, 
        "species": species, 
        "positions": positions_abs, 
        "forces": forces_np
    }

# Execute
data = parse_manual_castep('pdos_results/frame_000.castep')
print(f"Parsed {len(data['species'])} atoms with energy {data['energy']} eV.")

Parsed 72 atoms with energy -12989.25574371 eV.


In [2]:
# 1. Print the header info
print(f"{'Species':<10} | {'Position (x, y, z)':<35} | {'Force (fx, fy, fz)':<35}")
print("-" * 85)

# 2. Slice and display the first 3 atoms
for i in range(3):
    s = data['species'][i]
    pos = data['positions'][i]
    force = data['forces'][i]
    
    # Format for readability
    pos_str = f"{pos[0]:8.4f}, {pos[1]:8.4f}, {pos[2]:8.4f}"
    force_str = f"{force[0]:8.4f}, {force[1]:8.4f}, {force[2]:8.4f}"
    
    print(f"{s:<10} | {pos_str:<35} | {force_str:<35}")

# 3. Print the total Energy once more for confirmation
print("-" * 85)
print(f"Total Energy: {data['energy']} eV")

Species    | Position (x, y, z)                  | Force (fx, fy, fz)                 
-------------------------------------------------------------------------------------
H          |   0.7715,   0.9511,   0.9050        |   0.1769,   0.1159,   0.2656       
H          |  -1.8864,   0.0142,  -0.2756        |  -0.1761,  -0.0526,  -0.2301       
H          |  -0.3339,   9.1505,   2.0481        |   0.1744,   0.0284,   0.2030       
-------------------------------------------------------------------------------------
Total Energy: -12989.25574371 eV


In [3]:
import re
import numpy as np

def parse_manual_bands(filename):
    with open(filename, 'r') as f:
        lines = f.readlines()

    eigenvalues = []
    reading_eigenvalues = False

    for line in lines:
        # Clean tags
        clean = re.sub(r"\\", "", line).strip()
        
        # Identify the start of the eigenvalue block
        if "Spin component 1" in clean:
            reading_eigenvalues = True
            continue
        
        # Identify the end of the file or next k-point
        if reading_eigenvalues:
            if "K-point" in clean or not clean:
                # If we encounter a new k-point or empty line, stop
                if len(eigenvalues) >= 250:
                    reading_eigenvalues = False
                continue
            
            # Extract numerical value
            try:
                val = float(clean)
                eigenvalues.append(val)
            except ValueError:
                continue

    return np.array(eigenvalues)

# Execute
eigenvalues = parse_manual_bands('pdos_results/frame_000.bands')
print(f"Extracted {len(eigenvalues)} eigenvalues.")
print(f"First 5: {eigenvalues[:5]}")
print(f"Last 5: {eigenvalues[-5:]}")

Extracted 250 eigenvalues.
First 5: [-0.84342838 -0.84184237 -0.84105727 -0.84064642 -0.81494592]
Last 5: [0.6239658  0.6297868  0.63656978 0.63932035 0.64225084]


In [4]:
import numpy as np
import castepxbin

pdos_data = castepxbin.read_pdos_bin("pdos_results/frame_000.pdos_bin")

# 1. Setup dimensions
num_atoms = 72
num_bands = 250
num_orbitals = 204

# 2. Extract raw data from your castepxbin dictionary
weights = pdos_data['pdos_weights']  # shape (204, 250, 1, 1)
ion_indices = pdos_data['ion']       # shape (204,) - CASTEP uses 1-based indexing
species_list = pdos_data['species']  # To keep track of which atom is which

# 3. Initialize the Final Map: (Atoms x Eigenvalues)
# Each entry [i, j] is the total contribution of band j to atom i
atom_eigenvalue_map = np.zeros((num_atoms, num_bands))

# 4. Aggregate orbital weights into atomic weights
for orbital_idx in range(num_orbitals):
    # Convert CASTEP 1-based index to Python 0-based index
    atom_idx = ion_indices[orbital_idx] - 1
    
    # Add the contribution of all 250 bands for this specific orbital
    # we use [orbital_idx, :, 0, 0] to get the vector of 250 band weights
    atom_eigenvalue_map[atom_idx, :] += weights[orbital_idx, :, 0, 0]

print(f"Mapping complete. Final Shape: {atom_eigenvalue_map.shape}")

Mapping complete. Final Shape: (72, 250)


In [5]:
# 1. Print the header info
print(f"{'Species':<10} | {'Position (x, y, z)':<35} | {'Force (fx, fy, fz)':<35}")
print("-" * 85)

# 2. Slice and display the first 3 atoms
for i in range(3):
    s = data['species'][i]
    pos = data['positions'][i]
    force = data['forces'][i]
    
    # Format for readability
    pos_str = f"{pos[0]:8.4f}, {pos[1]:8.4f}, {pos[2]:8.4f}"
    force_str = f"{force[0]:8.4f}, {force[1]:8.4f}, {force[2]:8.4f}"
    
    print(f"{s:<10} | {pos_str:<35} | {force_str:<35}")

# 3. Print the total Energy once more for confirmation
print("-" * 85)
print(f"Total Energy: {data['energy']} eV")

Species    | Position (x, y, z)                  | Force (fx, fy, fz)                 
-------------------------------------------------------------------------------------
H          |   0.7715,   0.9511,   0.9050        |   0.1769,   0.1159,   0.2656       
H          |  -1.8864,   0.0142,  -0.2756        |  -0.1761,  -0.0526,  -0.2301       
H          |  -0.3339,   9.1505,   2.0481        |   0.1744,   0.0284,   0.2030       
-------------------------------------------------------------------------------------
Total Energy: -12989.25574371 eV


In [6]:
import glob
import os

# Define the range of frames
all_frames = []
frame_files = sorted(glob.glob('pdos_results/frame_*.castep'))

print(f"Found {len(frame_files)} CASTEP files. Starting batch processing...")

for castep_path in frame_files:
    # Get the base name (e.g., 'frame_000') to find matching .bands and .pdos_bin
    base_name = os.path.splitext(os.path.basename(castep_path))[0]
    folder = os.path.dirname(castep_path)
    
    bands_path = os.path.join(folder, f"{base_name}.bands")
    pdos_bin_path = os.path.join(folder, f"{base_name}.pdos_bin")
    
    # Check if all necessary files exist for this frame
    if not os.path.exists(bands_path) or not os.path.exists(pdos_bin_path):
        print(f"Skipping {base_name}: Missing .bands or .pdos_bin")
        continue

    try:
        # 1. Parse structural data
        frame_data = parse_manual_castep(castep_path)
        
        # 2. Parse eigenvalues
        # Note: Ensure parse_manual_bands returns the 250 array
        # (Modify the function to take the specific bands_path)
        frame_eigenvalues = parse_manual_bands(bands_path)
        
        # 3. Parse and map PDOS weights
        pdos_data = castepxbin.read_pdos_bin(pdos_bin_path)
        weights = pdos_data['pdos_weights']
        ions = pdos_data['ion']
        
        # Create the (72, 250) map for this specific frame
        current_map = np.zeros((72, 250))
        for orbital_idx in range(204):
            atom_idx = ions[orbital_idx] - 1
            current_map[atom_idx, :] += weights[orbital_idx, :, 0, 0]
        
        # Convert eigenvalues from Hartree to eV for the energy axis
        eigen_energies_ev = frame_eigenvalues * 27.2114  # Hartree -> eV
        
        # Store as a tuple: (structural_dict, eigenvalue_map, energy_axis_eV)
        all_frames.append((frame_data, current_map, eigen_energies_ev))
        
    except Exception as e:
        print(f"Error processing {base_name}: {e}")

print(f"Successfully processed {len(all_frames)} frames.")

Found 1001 CASTEP files. Starting batch processing...
Successfully processed 1001 frames.


In [7]:
def write_mace_dataset(filename, dataset):
    """
    Writes a multi-frame dataset to Extended XYZ format.
    dataset: List of tuples (data_dict, eigenvalue_map, eigen_energies_eV)

    eigen_energies_eV  – 1-D array of length 250 with the energy (eV) of
                         each band.  Stored once per frame in the comment
                         line so the NN can reconstruct the full eDoS.
    """
    with open(filename, 'w') as f:
        for frame_data, eigen_map, eigen_energies in dataset:
            num_atoms = len(frame_data['species'])
            lattice_str = " ".join(map(str, frame_data['lattice'].flatten()))
            properties = "species:S:1:pos:R:3:forces:R:3:eigen_weights:R:250"
            
            # Serialise the shared energy axis (one value per band, eV)
            energies_str = " ".join([f"{e:.6f}" for e in eigen_energies])
            
            # Write Number of Atoms
            f.write(f"{num_atoms}\n")
            
            # Write Comment Line
            # eigen_energies is stored as a quoted space-separated array so
            # that any Extended-XYZ reader can parse it as a frame property.
            comment = (
                f'Lattice="{lattice_str}" '
                f'Properties={properties} '
                f'energy={frame_data["energy"]:.8f} '
                f'eigen_energies="{energies_str}" '
                f'pbc="T T T"'
            )
            f.write(f"{comment}\n")
            
            # Write Atomic Data
            for i in range(num_atoms):
                s = frame_data['species'][i]
                px, py, pz = frame_data['positions'][i]
                fx, fy, fz = frame_data['forces'][i]
                w_str = " ".join([f"{w:.6f}" for w in eigen_map[i, :]])
                
                f.write(f"{s} {px:12.8f} {py:12.8f} {pz:12.8f} "
                        f"{fx:12.8f} {fy:12.8f} {fz:12.8f} "
                        f"{w_str}\n")

# Run the writer
write_mace_dataset('cellulose_full_dataset.xyz', all_frames)
print("Complete dataset saved to cellulose_full_dataset.xyz")

Complete dataset saved to cellulose_full_dataset.xyz


In [8]:
def extract_e0_values(filename):
    with open(filename, 'r') as f:
        content = f.read()
    
    # Regex to find element name and its corresponding pseudo atomic energy
    # Matches: Pseudo atomic calculation performed for [Element] ... total energy of [Energy] eV
    pattern = r"Pseudo atomic calculation performed for (\w+)[\s\S]+?total energy of\s+(-?\d+\.\d+) eV"
    matches = re.findall(pattern, content)
    
    # Store in a dictionary: {'H': -12.4813, 'C': -148.5211, ...}
    e0_dict = {elem: float(en) for elem, en in matches}
    return e0_dict

# Usage
e0_references = extract_e0_values('pdos_results/frame_1000.castep')
print("Extracted E0 (eV):", e0_references)

Extracted E0 (eV): {'H': -12.4813, 'C': -148.5211, 'O': -432.0344}


In [9]:
import ase.db
from ase import Atoms
from ase.calculators.singlepoint import SinglePointCalculator
import numpy as np

def create_orb_database(db_filename, dataset):
    """
    dataset: List of tuples (frame_data, current_map, eigen_energies_ev)
    """
    db = ase.db.connect(db_filename)
    
    for frame_data, eigen_map, eigen_energies in dataset:
        # 1. Create the Atoms object
        atoms = Atoms(
            symbols=frame_data['species'],
            positions=frame_data['positions'],
            cell=frame_data['lattice'],
            pbc=True
        )
        
        # 2. Attach standard training targets (Energy and Forces)
        calc = SinglePointCalculator(
            atoms=atoms,
            energy=frame_data['energy'],
            forces=frame_data['forces']
        )
        atoms.calc = calc
        
        # 3. Add Secondary Data for Orb-v3 Fine-tuning
        # 'eigen_energies' is global (per frame), 'eigen_weights' is per-atom
        data_payload = {
            'eigenvalues': eigen_energies, # Shape: (250,)
            'weights': eigen_map            # Shape: (72, 250)
        }
        
        # 4. Write to DB with the data payload
        # This makes 'eigenvalues' and 'weights' accessible to the GNN
        db.write(atoms, data=data_payload)

# Execute conversion
create_orb_database('cellulose_finetuning.db', all_frames)
print(f"Created ASE database with {len(all_frames)} structures.")

Created ASE database with 1001 structures.
